# Fake News Detection Using Machine Learning

## Objective
Develop a machine learning model capable of classifying news articles as **Fake** or **Real** using Natural Language Processing (NLP) techniques.

This notebook demonstrates the complete machine learning workflow:
1. Data Collection & Loading
2. Data Cleaning & Integration
3. Exploratory Data Analysis (EDA) & Visualization
4. Text Preprocessing
5. Feature Extraction (TF-IDF Vectorization)
6. Model Training & Evaluation (comparing Logistic Regression, Naive Bayes, Support Vector Machines, and Random Forest)
7. Model Selection & Asset Serialization

### Step 1: Import Required Libraries

In [ ]:
import os
import re
import string
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tqdm import tqdm
from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]

print("Libraries successfully imported!")

### Step 2: Load Datasets

In [ ]:
# Define dataset paths
fake_path = "../dataset/Fake.csv"
true_path = "../dataset/True.csv"

fake_df = pd.read_csv(fake_path)
true_df = pd.read_csv(true_path)

print(f"Fake News dataset loaded: {fake_df.shape} articles")
print(f"Real News dataset loaded: {true_df.shape} articles")

### Step 3: Merge, Shuffle, and Reset Index

In [ ]:
# Add labels
fake_df['label'] = 0
true_df['label'] = 1

# Merge datasets
df = pd.concat([fake_df, true_df], ignore_index=True)

# Shuffle dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Merged Dataset shape: {df.shape}")
df.head()

### Step 4: Exploratory Data Analysis (EDA)
Let's look at the label distribution, missing values, duplicates, and article length distribution.

In [ ]:
# Data Types & Info
print("--- Data Types & Info ---")
print(df.info())

# Missing Values
print("\n--- Missing Values ---")
print(df.isnull().sum())

# Duplicate Values
print(f"\nDuplicate rows based on content: {df.duplicated(subset=['title', 'text']).sum()}")
df = df.drop_duplicates(subset=['title', 'text']).reset_index(drop=True)
print(f"Shape after dropping duplicates: {df.shape}")

#### Visualizing Label Distribution

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(x='label', data=df, palette={0: "#e74c3c", 1: "#2ecc71", "0": "#e74c3c", "1": "#2ecc71"})
ax.set_xticklabels(['Fake (0)', 'True (1)'])
plt.title('Distribution of Fake vs Real News')
plt.xlabel('News Type')
plt.ylabel('Number of Articles')
plt.show()

#### Visualizing Article Word Length Distribution

In [ ]:
df['text_len'] = df['text'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(10, 5))
sns.kdeplot(df[df['label'] == 0]['text_len'], label='Fake News', shade=True, color="#e74c3c")
sns.kdeplot(df[df['label'] == 1]['text_len'], label='Real News', shade=True, color="#2ecc71")
plt.xlim(0, 1500)
plt.title('Article Word Length Distribution')
plt.xlabel('Word Count')
plt.ylabel('Density')
plt.legend()
plt.show()

#### Word Clouds & Common Words in Headlines

In [ ]:
fake_titles = " ".join(df[df['label'] == 0]['title'].astype(str).tolist()[:1000])
true_titles = " ".join(df[df['label'] == 1]['title'].astype(str).tolist()[:1000])

wc_fake = WordCloud(width=800, height=400, background_color='white', colormap='Reds').generate(fake_titles)
wc_true = WordCloud(width=800, height=400, background_color='white', colormap='Greens').generate(true_titles)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(wc_fake, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('Word Cloud - Fake News Titles', fontsize=14)

axes[1].imshow(wc_true, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('Word Cloud - Real News Titles', fontsize=14)
plt.tight_layout()
plt.show()

### Step 5: Text Preprocessing
We create a reusable cleaning pipeline that performs lowercasing, HTML/URL stripping, punctuation and number removal, tokenization, stopword filtering, and lemmatization.

In [ ]:
# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    # Lowercase
    text = text.lower()
    # Remove HTML
    text = re.sub(r'<[^>]*>', ' ', text)
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    # Remove digits
    text = re.sub(r'\d+', ' ', text)
    # Remove punctuation
    text = text.translate(str.maketrans(string.punctuation, ' ' * len(string.punctuation)))
    # Tokenize
    tokens = word_tokenize(text)
    # Stopwords & Lemmatization
    processed = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words and len(w) > 2]
    return " ".join(processed)

# Downsample slightly for interactive notebook responsiveness
df_sample = df.sample(n=10000, random_state=42).reset_index(drop=True)
df_sample['full_text'] = df_sample['title'] + " " + df_sample['text']

tqdm.pandas(desc="Cleaning text")
df_sample['clean_text'] = df_sample['full_text'].progress_apply(preprocess_text)
print("Preprocessing completed successfully.")

### Step 6 & 7: Feature Extraction & Train-Test Split
We use standard TF-IDF representations and divide our dataset into an 80/20 train/test split.

In [ ]:
X = df_sample['clean_text']
y = df_sample['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TF-IDF Vectorizer
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2), stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"X_train_vec shape: {X_train_vec.shape}")
print(f"X_test_vec shape: {X_test_vec.shape}")

### Step 8, 9 & 10: Model Training, Evaluation, and Comparison
We train Logistic Regression, Naive Bayes, Linear Support Vector Classification (SVC), and Random Forest, and print a consolidated comparison table.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Multinomial Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=50, max_depth=15, random_state=42, n_jobs=-1)
}

results = []

for name, clf in models.items():
    print(f"Training {name}...")
    clf.fit(X_train_vec, y_train)
    y_pred = clf.predict(X_test_vec)
    
    # Metrics calculation
    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary')
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1
    })
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'])
    plt.title(f"{name} - Confusion Matrix")
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()
    print(f"Classification Report for {name}:\n", classification_report(y_test, y_pred))

# Display Comparison Summary
df_results = pd.DataFrame(results)
print("\n--- Model Performance Comparison Table ---")
df_results

### Step 11: Save Best Model & Vectorizer

In [ ]:
# Find best model name by F1-Score
best_row = df_results.loc[df_results['F1-Score'].idxmax()]
best_name = best_row['Model']
best_model_instance = models[best_name]

print(f"Best model is {best_name} with F1-Score of {best_row['F1-Score']:.4f}")

# Save model assets
os.makedirs("../models", exist_ok=True)
joblib.dump(best_model_instance, "../models/model.pkl")
joblib.dump(vectorizer, "../models/vectorizer.pkl")

print("Model and Vectorizer assets serialized to '../models/' directory.")

### Step 12: Custom Prediction Example

In [ ]:
# Simple testing function
def custom_prediction(text):
    cleaned = preprocess_text(text)
    vec = vectorizer.transform([cleaned])
    pred = best_model_instance.predict(vec)[0]
    label = "REAL NEWS" if pred == 1 else "FAKE NEWS"
    
    if hasattr(best_model_instance, "predict_proba"):
        confidence = best_model_instance.predict_proba(vec)[0][pred] * 100
    else:
        confidence = 100.0 # Standard LinearSVC fallback
    
    print(f"Input Text: {text[:100]}...")
    print(f"Prediction: {label} (Confidence: {confidence:.2f}%)\n")

# Test cases
custom_prediction("President signs new bipartisan climate bill into law today in Washington.")
custom_prediction("Shocking: Aliens landed on the White House lawn yesterday and spoke directly to the President!")